<a href="https://colab.research.google.com/github/duckytej/Ai_/blob/main/chapter_appendix-tools-for-deep-learning/jupyter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

A.I.

In [2]:
!pip install unsloth
!pip install peft
!pip install transformers
!pip install accelerate
!pip install bitsandBytes
!pip install datasets

In [3]:
!pip install GPUtil
import unsloth
import torch
import GPUtil
import os
GPUtil.showUtilization()
if torch.cuda.is_available():
  print("available")
else:
  print("not available")
os.environ["cuda_device_order"]="pci_bus_id"
os.environ["cuda_visible_devices"]="0"
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, LlamaTokenizer
from huggingface_hub import notebook_login
from datasets import load_dataset
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

if "colab_gpu" in os.environ:
  from google.colab import output
  output.enable_custom_widget_manager()

  Preparing metadata (setup.py) ... done
  Created wheel for GPUtil: filename=GPUtil-1.4.0-py3-none-any.whl size=7392 sha256=6e8d5ee21b9da930a4a8f79d461a482a0bd312f7c7a8951c241002ee770e936e
  Stored in directory: /root/.cache/pip/wheels/92/a8/b7/d8a067c31a74de9ca252bbe53dea5f896faabd25d55f541037
Successfully built GPUtil
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
| ID | GPU | MEM |
------------------
|  0 |  0% |  1% |
available


In [8]:
if"colab_gpu" in os.environ:
  !huggingface-cli notebook_login
else:
  notebook_login()

# Task
Configure Unsloth, load the `PantheonUnbound/Satyr-V0.1-4B` model in 4-bit quantization, and load its corresponding tokenizer.

## Configure Unsloth and Load Model

### Subtask:
This cell will set up Unsloth, load the `PantheonUnbound/Satyr-V0.1-4B` model in 4-bit quantization, and configure it for LoRA training. It also loads the corresponding tokenizer.


In [9]:
from unsloth import FastLanguageModel
import torch

# 1. Use the repo that actually has the Safetensors + Config
model_name = "TheDrummer/Gemmasutra-Mini-2B-v1"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = 2048,
    dtype = None,           # Auto-detect (Float16/BFloat16)
    load_in_4bit = True,    # Essential for saving memory
    trust_remote_code = True # Required for pruned/custom architectures
)

# 2. Add the LoRA adapters to make it trainable
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

print("Model Loaded Successfully!")

Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.3.4: Fast Gemma2 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Unsloth: Will load TheDrummer/Gemmasutra-Mini-2B-v1 as a legacy tokenizer.


Model Loaded Successfully!


# Task
Load the `ultimate_ig.jsonl` dataset, split it into 90% for training and 10% for validation, and then format the dataset for use with the Unsloth SFTTrainer.

## Load Dataset with Split

### Subtask:
Load the `ultimate_ig.jsonl` file into a Hugging Face Dataset object and create a 90/10 training and validation split.


In [10]:
from datasets import load_dataset

# Load the dataset from the JSONL file
dataset = load_dataset('json', data_files='ultimate_ig.jsonl')

# Create a 90/10 training and validation split
dataset = dataset['train'].train_test_split(test_size=0.1)

print("Dataset loaded and split successfully!")
print(dataset)

Dataset loaded and split successfully!
DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 451
    })
    test: Dataset({
        features: ['messages'],
        num_rows: 51
    })
})


# Task
Format the loaded dataset into the format expected by the Unsloth SFTTrainer, concatenating the 'messages' into a single 'text' column, formatted using the tokenizer's chat template.

## Format Dataset for Training

### Subtask:
Transform the loaded dataset into the format expected by the Unsloth SFTTrainer. This involves concatenating the 'messages' into a single 'text' column, formatted using the tokenizer's chat template. The `SFTTrainer` will then ensure training occurs only on assistant responses.


In [20]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template, train_on_responses_only
from jinja2.exceptions import TemplateError
import os

# Load dataset
dataset = load_dataset("json", data_files="ultimate_ig.jsonl")["train"]
dataset = dataset.train_test_split(test_size=0.1)

# Force Gemma chat template
tokenizer = get_chat_template(tokenizer, chat_template="gemma")

# -------------------------
# CLEAN CONVERSATIONS
# -------------------------
def clean_conversations(examples):
    cleaned = []

    for messages in examples["messages"]:
        valid_messages = [
            msg for msg in messages
            if isinstance(msg, dict)
            and "role" in msg
            and "content" in msg
            and msg["content"].strip() != ""
        ]

        if len(valid_messages) < 2:
            cleaned.append(None)
            continue

        # Ensure first message is user
        if valid_messages[0]["role"] == "assistant":
            valid_messages[0]["role"] = "user"

        if valid_messages[0]["role"] != "user":
            cleaned.append(None)
            continue

        # enforce alternation
        expected_role = "user"
        convo = []
        valid = True

        for msg in valid_messages:
            if msg["role"] == expected_role:
                convo.append(msg)
                expected_role = "assistant" if expected_role == "user" else "user"
            else:
                valid = False
                break

        if valid and len(convo) >= 2:
            cleaned.append(convo)
        else:
            cleaned.append(None)

    return {"messages": cleaned}

dataset = dataset.map(
    clean_conversations,
    batched=True,
    num_proc=os.cpu_count(),
)

# Remove broken conversations
dataset = dataset.filter(lambda x: x["messages"] is not None)

# -------------------------
# APPLY CHAT TEMPLATE
# -------------------------
def formatting_prompts_func(examples):
    texts = []

    for convo in examples["messages"]:
        try:
            text = tokenizer.apply_chat_template(
                convo,
                tokenize=False,
                add_generation_prompt=False
            )
            texts.append(text)
        except TemplateError:
            texts.append("")

    return {"text": texts}

dataset = dataset.map(
    formatting_prompts_func,
    batched=True,
    num_proc=os.cpu_count(),
)

dataset = dataset.filter(lambda x: x["text"] != "")

# -------------------------
# TRAIN ONLY ON RESPONSES
# -------------------------

print("Dataset ready for SFTTrainer!")
print(dataset)

Dataset ready for SFTTrainer!
DatasetDict({
    train: Dataset({
        features: ['messages', 'text'],
        num_rows: 343
    })
    test: Dataset({
        features: ['messages', 'text'],
        num_rows: 38
    })
})


## Configure Training Arguments

### Subtask:
Define the training parameters, such as epochs, learning rate, and batch size, using `TrainingArguments` from the `transformers` library.


**Reasoning**:
I need to define the training parameters using `TrainingArguments` from the `transformers` library with the specified values. This involves importing `TrainingArguments` and instantiating it with the provided parameters, including checking for `bf16` support using `torch`.



In [17]:
from transformers import TrainingArguments
import torch

training_arguments = TrainingArguments(
    per_device_train_batch_size = 2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps = 4,
    warmup_steps = 5,
    num_train_epochs=2,
    learning_rate = 2e-4,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 2,
    optim = "adamw_8bit",
    seed = 3407,
    output_dir = "outputs",

    eval_strategy="steps",                # evaluate every `eval_steps`
    eval_steps=10,                               # adjust as needed
    logging_strategy="steps",

    save_strategy="steps",
    save_steps=20,                                # save every 20 steps
    save_total_limit=2,                           # keep only last 2 checkpoints
    load_best_model_at_end=True,

    metric_for_best_model="eval_loss",            # we care about eval loss
    greater_is_better=False,

)

print("Training arguments defined successfully!")

Training arguments defined successfully!


## Initialize and Train SFTTrainer

### Subtask:
Initialize the `SFTTrainer` from `trl` with the loaded model, tokenizer, formatted dataset, and training arguments. Then, start the training process.


**Reasoning**:
I need to initialize the `SFTTrainer` and start the training process as per the subtask. This involves importing `SFTTrainer`, setting `max_seq_length` and `packing`, and then instantiating and calling the `train()` method.



In [21]:
from trl import SFTTrainer

# Set max_seq_length for the SFTTrainer
max_seq_length = 2048 # This should match the model's max_seq_length

# Initialize SFTTrainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset['train'],
    eval_dataset = dataset['test'],
    args = training_arguments,
    max_seq_length = max_seq_length,
    packing = False,
)
trainer = train_on_responses_only(
    trainer,
    tokenizer=tokenizer,
    instruction_part="<start_of_turn>user\n",
    response_part="<start_of_turn>model\n",
)

# Start training
trainer.train()

print("SFTTrainer initialized and training started!")

Unsloth: Tokenizing ["text"] (num_proc=3):   0%|          | 0/343 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=3):   0%|          | 0/38 [00:00<?, ? examples/s]

Map (num_proc=3):   0%|          | 0/343 [00:00<?, ? examples/s]

Filter (num_proc=3):   0%|          | 0/343 [00:00<?, ? examples/s]

Map (num_proc=3):   0%|          | 0/38 [00:00<?, ? examples/s]

Filter (num_proc=3):   0%|          | 0/38 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 343 | Num Epochs = 2 | Total steps = 86
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 20,766,720 of 2,635,108,608 (0.79% trained)
--- Logging error ---
Traceback (most recent call last):
  File "/usr/lib/python3.12/logging/__init__.py", line 1160, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 999, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 703, in format
    record.message = record.getMessage()
                     ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 392, in getMessage
    msg = msg % self.args
          ~~~~^~~~~~~~~~~
TypeError: not all arguments converted d

Step,Training Loss,Validation Loss
10,3.942209,3.792560
20,3.574231,3.631028
30,3.551228,3.556337
40,3.407461,3.513011
50,3.389895,3.489870
60,3.117899,3.471418
70,3.323842,3.458678
80,3.327898,3.452678


Unsloth: Not an error, but Gemma2ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


SFTTrainer initialized and training started!


# Task
Verify the fine-tuned model by loading the adapter weights and merging them with the base model, and then save the complete fine-tuned model and its tokenizer for future inference.

In [29]:
!pip install gguf
import os
# Ensure we have the latest llama.cpp requirements if needed
!pip install -U "huggingface_hub[cli]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 612.9/612.9 kB 57.1 MB/s eta 0:00:00
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.1
    Uninstalling hf-xet-1.3.1:
      Successfully uninstalled hf-xet-1.3.1
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.5.0
    Uninstalling huggingface_hub-1.5.0:
      Successfully uninstalled huggingface_hub-1.5.0


In [31]:
import torch
import gc
import os

# 1. Memory Cleanup
if 'trainer' in globals(): del trainer
gc.collect()
torch.cuda.empty_cache()

# 2. Fix Tokenizer for Gemma 2 GGUF export
# Gemma 2 often requires padding_side to be right for the conversion script to behave
tokenizer.padding_side = 'right'

# 3. Attempt conversion with explicit GGUF settings
try:
    # We use the internal unsloth export which handles the llama.cpp pathing
    model.save_pretrained_gguf(
        "model_gguf_final_v2",
        tokenizer,
        quantization_method = "q4_k_m",
    )
    print("GGUF conversion completed successfully!")
except Exception as e:
    print(f"Conversion still failing. Error: {e}")
    print("Checking for conversion log...")
    # If it fails, we check the directory to see if any temp files were made
    if os.path.exists("model_gguf_final_v2"):
        print("Current files in output dir:", os.listdir("model_gguf_final_v2"))

Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### Your chat template has a BOS token. We shall remove it temporarily.


Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...



Unsloth: Copying 2 files from cache to `model_gguf_final_v2`:   0%|          | 0/2 [00:00<?, ?it/s]
Unsloth: Copying 2 files from cache to `model_gguf_final_v2`:  50%|█████     | 1/2 [01:12<01:12, 72.81s/it]
Unsloth: Copying 2 files from cache to `model_gguf_final_v2`: 100%|██████████| 2/2 [01:16<00:00, 38.07s/it]


Successfully copied all 2 files from cache to `model_gguf_final_v2`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.



Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [00:00<00:00, 15033.35it/s]

Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [01:25<00:00, 42.93s/it]


Unsloth: Merge process complete. Saved to `/content/model_gguf_final_v2`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Conversion still failing. Error: Unsloth: GGUF conversion failed: Unsloth: Failed to convert model to GGUF with command `/usr/bin/python3 /root/.unsloth/llama.cpp/unsloth_convert_hf_to_gguf.py --outfile Gemmasutra-Mini-2B-v1.F16.gguf --outtype f16 --split-max-size 50G model_gguf_final_v2`: Command '['/usr/bin/python3', '/root/.unsloth/llama.cpp/unsloth_convert_hf_to_

In [32]:
import torch
import gc

# 1. Final Memory Sweep
if 'trainer' in globals(): del trainer
gc.collect()
torch.cuda.empty_cache()

# 2. Save the model in merged 16-bit format (Standard HF)
# This bypasses the GGUF script while preserving your fine-tuning
try:
    model.save_pretrained_merged(
        "final_merged_model",
        tokenizer,
        save_method = "merged_16bit",
    )
    print("Model successfully merged and saved to 'final_merged_model' folder!")
except Exception as e:
    print(f"Merged save failed: {e}")

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...



Unsloth: Copying 2 files from cache to `final_merged_model`:   0%|          | 0/2 [00:00<?, ?it/s]
Unsloth: Copying 2 files from cache to `final_merged_model`:  50%|█████     | 1/2 [01:18<01:18, 78.05s/it]
Unsloth: Copying 2 files from cache to `final_merged_model`: 100%|██████████| 2/2 [01:22<00:00, 41.11s/it]


Successfully copied all 2 files from cache to `final_merged_model`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.



Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [00:00<00:00, 5457.78it/s]

Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [02:06<00:00, 63.32s/it]


Unsloth: Merge process complete. Saved to `/content/final_merged_model`
Model successfully merged and saved to 'final_merged_model' folder!


In [33]:
import os
if os.path.exists("final_merged_model"):
    print("Files saved successfully:")
    print(os.listdir("final_merged_model"))
else:
    print("Save directory not found.")

Files saved successfully:
['model-00002-of-00002.safetensors', 'tokenizer_config.json', 'tokenizer.json', 'model.safetensors.index.json', 'config.json', 'chat_template.jinja', '.cache', 'model-00001-of-00002.safetensors']


In [2]:
import shutil
import os

# Define the folder to zip and the output filename
folder_to_zip = 'final_merged_model'
output_filename = 'final_merged_model_download.zip'

if os.path.exists(folder_to_zip):
    print(f'Zipping {folder_to_zip}...')
    shutil.make_archive(output_filename.replace('.zip', ''), 'zip', folder_to_zip)
    print(f'Created {output_filename}')
else:
    print(f'Error: {folder_to_zip} not found.')

Error: final_merged_model not found.


In [1]:
from google.colab import files
import os

zip_path = 'final_merged_model_download.zip'
if os.path.exists(zip_path):
    files.download(zip_path)
else:
    print('Zip file not found. Please run the zipping cell first.')

Zip file not found. Please run the zipping cell first.


In [ ]:
from google.colab import static_widget_proxy
from IPython.display import display, HTML

# This creates a simple HTML link that can sometimes bypass the buffer hang
print("If the automatic download doesn't start, use the sidebar:")
print("1. Click the Folder icon on the left.")
print("2. Right-click 'final_merged_model_download.zip'.")
print("3. Select 'Download'.")

display(HTML(f'<a href="/files/final_merged_model_download.zip" download>Click here to attempt manual browser download of the zip</a>'))

# Task
Test the fine-tuned model by loading the merged model and tokenizer from the "final_model" directory and then generating text with a sample prompt.

## Load Fine-tuned Model and Tokenizer

### Subtask:
Load the merged fine-tuned model and its corresponding tokenizer from the 'final_model' directory. This will allow us to use the model for inference.


**Reasoning**:
I need to load the fine-tuned model and its tokenizer from the 'final_model' directory using `FastLanguageModel.from_pretrained` with 4-bit quantization as specified.



In [38]:
from unsloth import FastLanguageModel

# Load the fine-tuned model and tokenizer from the correct directory
final_merged_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "final_merged_model", # Changed from 'final_model' to match the saved folder
    load_in_4bit = True,
)

print("Fine-tuned model and tokenizer loaded successfully from 'final_merged_model'!")

==((====))==  Unsloth 2026.3.4: Fast Gemma2 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Unsloth: Will load final_merged_model as a legacy tokenizer.


Fine-tuned model and tokenizer loaded successfully from 'final_merged_model'!


# Task
Generate a response from the fine-tuned model for the prompt "good morning" and explicitly print the generated response. Upon successful generation, summarize the successful generation of the model's response and indicate readiness for further testing or deployment.

## Modify Prompt and Generate Response

### Subtask:
Revise the prompt to a simple 'good morning' user message and regenerate a response using the loaded model. The generated response will be explicitly printed.


**Reasoning**:
I need to define the prompt, format it using the tokenizer's chat template, tokenize it, generate a response from the model, decode the response, and then print it as per the instructions.



In [48]:
import torch

# 1. Define the prompt
messages = [
    {"role": "user", "content": "what is your name"},
]

# 2. Format with template
formatted_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_special_tokens=True,
    add_generation_prompt=True
)

# 3. Tokenize
inputs = tokenizer(
    formatted_prompt,
    return_tensors="pt"
).to("cuda")

# 4. Generate with REPETITION PENALTY to fix the 'i am i am' issue
input_len = inputs.input_ids.shape[1]
outputs = model.generate(**inputs,
    max_new_tokens=64,
    use_cache=True,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.2, # Added to stop word repetition
)

# 5. Decode
response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)

# 6. Print result
print("Assistant Response:")
print(response)

Assistant Response:
my name is anushka
yes
um
um
i am
i am


In [ ]:
import json

# Let's look at the first 10 assistant responses in the dataset
count = 0
with open('ultimate_ig.jsonl', 'r') as f:
    for line in f:
        data = json.loads(line)
        for msg in data.get('messages', []):
            if msg.get('role') == 'assistant':
                print(f"Assistant sample {count+1}: {msg.get('content')[:100]}...")
                count += 1
        if count >= 10: break